# 01 — Citibike + Weather Data Preparation

Builds the daily, per-station Citibike forecasting tables (`_source` → `_prepped`) with NOAA
GSOD weather covariates, station metadata, and calendar features — mirroring the statmike
BigQuery time-series prep pattern. SQL is generated by `geaptimes.data.queries` from the
validated `config/base_config.yaml`.

## Prerequisites
- Authenticated credentials: `gcloud auth application-default login`
- `GCP_PROJECT` environment variable set
- Launch with `uv run jupyter lab` (so the `geaptimes` package is importable)
- Resources provisioned: `uv run python ../scripts/setup_gcp.py --config ../config/base_config.yaml`

> The Citibike public table is **frozen (~2018)** and `citibike_stations` is a **current
> snapshot** — see `docs/notes/forecasting-covariate-and-model-constraints.md`.

In [ ]:
import os

import matplotlib.pyplot as plt
from google.cloud import bigquery

from geaptimes.data.queries import (
    build_prepped_query,
    build_source_query,
    table_id,
)
from geaptimes.naming import table_names
from geaptimes.schemas import ExperimentConfig

assert os.environ.get("GCP_PROJECT"), "Set GCP_PROJECT before running."
cfg = ExperimentConfig.from_yaml("../config/base_config.yaml")
data = cfg.data
names = table_names(cfg)
client = bigquery.Client(project=cfg.project.id)
print("project:", cfg.project.id, "| dataset:", cfg.project.bq_dataset)

## 1. Inspect the frozen Citibike window
Confirm the table's actual date span and volume before defining splits.

In [ ]:
span = client.query(f"""
    SELECT MIN(DATE(starttime)) AS min_date,
           MAX(DATE(starttime)) AS max_date,
           COUNT(*) AS n_trips
    FROM `{data.trips_table}`
""").to_dataframe()
span

## 2. Confirm the NOAA GSOD weather station
Verify the configured station (LaGuardia `725030`/`14732` by default) exists and has coverage.

In [ ]:
weather_check = client.query(f"""
    SELECT COUNT(*) AS n_days,
           MIN(PARSE_DATE('%Y%m%d', CONCAT(year, mo, da))) AS min_date,
           MAX(PARSE_DATE('%Y%m%d', CONCAT(year, mo, da))) AS max_date
    FROM `{data.weather_dataset}.gsod*`
    WHERE _TABLE_SUFFIX BETWEEN '2013' AND '2019'
      AND stn = '{data.weather.station_usaf}' AND wban = '{data.weather.station_wban}'
""").to_dataframe()
weather_check

## 3. Build the `_source` table
Daily aggregation per top-N station with engineered covariates, weather, station metadata,
and calendar features. Print the generated SQL for transparency, then run it.

In [ ]:
source = table_id(cfg.project, names["source"])
source_sql = build_source_query(data, source)
print(source_sql)
client.query(source_sql).result()
print("built", source)

## 4. Station name-join coverage
Top-N station names join to the `citibike_stations` snapshot by name. Report unmatched
stations (their `capacity`/metadata will be null).

In [ ]:
coverage = client.query(f"""
    SELECT t.{data.series_column} AS series,
           m.capacity IS NOT NULL AS has_metadata
    FROM (SELECT DISTINCT {data.series_column} FROM `{source}`) t
    LEFT JOIN `{data.stations_table}` m
      ON m.name = t.{data.series_column}
    ORDER BY has_metadata, series
""").to_dataframe()
print("unmatched:", (~coverage['has_metadata']).sum(), "/", len(coverage))
coverage

## 5. Build the `_prepped` table (splits)
Add TRAIN/VALIDATE/TEST by date range off the global max date (TEST=14, VALIDATE=14).

In [ ]:
prepped = table_id(cfg.project, names["prepped"])
prepped_sql = build_prepped_query(data, source, prepped)
print(prepped_sql)
client.query(prepped_sql).result()

split_counts = client.query(f"""
    SELECT {data.series_column} AS series,
           COUNTIF(splits='TRAIN') AS train,
           COUNTIF(splits='VALIDATE') AS validate,
           COUNTIF(splits='TEST') AS test
    FROM `{prepped}`
    GROUP BY series ORDER BY series
""").to_dataframe()
split_counts

## 6. Visualize per-series ranges with split bands

In [ ]:
ranges = client.query(f"""
    SELECT {data.series_column} AS series,
           MIN({data.time_column}) AS d0, MAX({data.time_column}) AS d1
    FROM `{prepped}` GROUP BY series ORDER BY series
""").to_dataframe()

test_len = data.splits.test_length
val_plus_test = data.splits.test_length + data.splits.validate_length
key = client.query(f"""
    SELECT DATE_SUB(MAX({data.time_column}), INTERVAL {val_plus_test} DAY) AS val_start,
           DATE_SUB(MAX({data.time_column}), INTERVAL {test_len} DAY) AS test_start,
           MAX({data.time_column}) AS end_date
    FROM `{prepped}`
""").to_dataframe().iloc[0]

fig, ax = plt.subplots(figsize=(16, 8))
for i, row in ranges.iterrows():
    ax.plot([row['d0'], row['d1']], [i, i], 'o--', color='steelblue', markersize=3)
ax.axvspan(key['val_start'], key['test_start'], color='gold', alpha=0.3, label='validate')
ax.axvspan(key['test_start'], key['end_date'], color='tomato', alpha=0.3, label='test')
ax.set_yticks(range(len(ranges)))
ax.set_yticklabels(ranges['series'], fontsize=7)
ax.set_title('Per-station date ranges with validate/test bands')
ax.legend(loc='lower left')
plt.tight_layout()
plt.show()

## Summary
`<dataset>.citibike_daily_prepped` is the clean, split-labeled table the Stage 2 model
wrappers consume (via `ExperimentConfig`). Covariate roles and per-model usage are documented
in `docs/notes/forecasting-covariate-and-model-constraints.md`.